# When Memory vs Retrieval Alone

Teams often ask: *"Do we need agent memory if we already have RAG?"* The answer depends on **what kind of knowledge** the agent must use.

| Capability | **Retrieval (RAG)** | **Memory** |
|------------|---------------------|------------|
| **Source** | Shared corpus (policies, docs, wiki) | Per-user / per-thread state |
| **Updates** | Re-index documents | Write after each turn |
| **Scope** | Same for all users | Private to user or session |
| **Examples** | Return policy, API docs | "My name is Alice", last order ID |
| **Audit** | Cite chunk IDs | Memory write log + TTL |

**Retrieval alone** answers *"What does ShopCo policy say?"*  
**Memory alone** answers *"What did this customer tell us five minutes ago?"*  
**Both** answer *"Given Alice's open refund on ORD-1001, what does policy allow?"*

This notebook implements three assistants on **ShopCo Support** — retrieval-only, memory-only, and hybrid — and compares them on multi-turn conversations. Everything is plain Python.

In [ ]:
"""
ShopCo Support — memory vs retrieval comparison lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- Retrieval corpus (shared policies) ---

POLICY_DOCS: list[dict[str, str]] = [
    {
        "id": "pol-returns-01",
        "title": "Returns Policy",
        "text": "Customers may return items within 30 days of delivery. Refunds in 5-7 business days. Final-sale items cannot be returned.",
    },
    {
        "id": "pol-shipping-02",
        "title": "Shipping Policy",
        "text": "Free standard shipping on orders over $50. Tracking emailed when the carrier scans the package.",
    },
    {
        "id": "pol-billing-03",
        "title": "Billing Policy",
        "text": "Charges appear as SHOPCO* on statements. Disputes require order ID.",
    },
]

ORDERS_DB: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "customer": "alice@example.com"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "customer": "bob@example.com"},
}

print(f"Corpus: {len(POLICY_DOCS)} policies | Orders: {len(ORDERS_DB)}")

---

## 1. Definitions — Do Not Conflate Them

```
  RETRIEVAL (RAG)                    MEMORY
  ─────────────────                  ──────
  External index                     Agent-owned store
  Authoritative docs                 User/session facts
  Read-mostly                        Read + write each turn
  "What is policy?"                  "Who is this user?"
```

Putting user-specific facts into a vector index is an **anti-pattern** — it pollutes shared retrieval and creates privacy risk. Storing policy PDFs only in chat memory is also wrong — memory is not a document CMS.

---

## 2. Policy Retriever (Shared Knowledge)

In [ ]:
@dataclass
class RetrievalHit:
    doc_id: str
    title: str
    text: str
    score: float


def retrieve_policies(query: str, top_k: int = 2) -> list[RetrievalHit]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored: list[tuple[float, dict[str, str]]] = []
    for doc in POLICY_DOCS:
        hay = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for t in terms if t in hay) if terms else 0.0
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [
        RetrievalHit(d["id"], d["title"], d["text"], s)
        for s, d in scored[:top_k]
    ]


print(retrieve_policies("return window")[0].doc_id)

---

## 3. Agent Memory Store (Per-User / Per-Thread)

In [ ]:
class MemoryKind(str, Enum):
    SEMANTIC = "semantic"    # stable facts: name, plan tier
    EPISODIC = "episodic"    # events: mentioned order, prior question
    WORKING = "working"      # current thread scratch


@dataclass
class MemoryRecord:
    key: str
    value: str
    kind: MemoryKind
    user_id: str
    thread_id: str
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


class AgentMemoryStore:
    """In-process memory — semantic + episodic + working layers."""

    def __init__(self) -> None:
        self._records: list[MemoryRecord] = []
        self._write_log: list[dict[str, Any]] = []

    def write(self, user_id: str, thread_id: str, key: str, value: str, kind: MemoryKind) -> None:
        # upsert by user+key for semantic; append for episodic
        if kind == MemoryKind.SEMANTIC:
            self._records = [r for r in self._records if not (r.user_id == user_id and r.key == key)]
        self._records.append(MemoryRecord(key, value, kind, user_id, thread_id))
        self._write_log.append({"user_id": user_id, "key": key, "kind": kind.value, "value": value})

    def read(self, user_id: str, thread_id: str | None = None) -> list[MemoryRecord]:
        out = [r for r in self._records if r.user_id == user_id]
        if thread_id:
            out = [r for r in out if r.thread_id == thread_id or r.kind == MemoryKind.SEMANTIC]
        return out

    def get(self, user_id: str, key: str) -> str | None:
        for r in reversed(self._records):
            if r.user_id == user_id and r.key == key:
                return r.value
        return None

    def format_for_prompt(self, user_id: str, thread_id: str) -> str:
        records = self.read(user_id, thread_id)
        if not records:
            return "(no user memory)"
        lines = [f"[{r.kind.value}] {r.key}={r.value}" for r in records[-8:]]
        return "\n".join(lines)


memory = AgentMemoryStore()
memory.write("alice", "t1", "customer_name", "Alice", MemoryKind.SEMANTIC)
memory.write("alice", "t1", "last_order_id", "ORD-1001", MemoryKind.EPISODIC)
print(memory.format_for_prompt("alice", "t1"))

---

## 4. Memory Extractor — What to Remember After Each Turn

Agents extract **write-worthy facts** from user messages — order IDs, names, preferences — not whole policy text.

In [ ]:
def extract_memory_writes(user_id: str, thread_id: str, message: str) -> list[tuple[str, str, MemoryKind]]:
    writes: list[tuple[str, str, MemoryKind]] = []
    m = re.search(r"ORD-[0-9]{4}", message.upper())
    if m:
        writes.append(("last_order_id", m.group(0), MemoryKind.EPISODIC))
    name_m = re.search(r"my name is ([A-Za-z]+)", message, re.I)
    if name_m:
        writes.append(("customer_name", name_m.group(1), MemoryKind.SEMANTIC))
    if "email me" in message.lower():
        email_m = re.search(r"[\w.-]+@[\w.-]+\.[A-Za-z]{2,}", message)
        if email_m:
            writes.append(("contact_email", email_m.group(0), MemoryKind.SEMANTIC))
    writes.append(("last_user_message", message[:120], MemoryKind.WORKING))
    return writes


def apply_memory_writes(store: AgentMemoryStore, user_id: str, thread_id: str, message: str) -> None:
    for key, value, kind in extract_memory_writes(user_id, thread_id, message):
        store.write(user_id, thread_id, key, value, kind)


demo_writes = extract_memory_writes("u1", "t1", "Hi, my name is Alice. Question about ORD-1001.")
print(demo_writes)

---

## 5. Retrieval-Only Assistant

Answers from policy index only. Ignores prior turns — no user context.

In [ ]:
@dataclass
class AssistantReply:
    answer: str
    sources: list[str]
    used_memory: bool
    used_retrieval: bool
    trace: list[str]


def retrieval_only_assistant(user_id: str, thread_id: str, message: str) -> AssistantReply:
    hits = retrieve_policies(message)
    if not hits:
        return AssistantReply(
            answer="I could not find a matching ShopCo policy.",
            sources=[], used_memory=False, used_retrieval=True,
            trace=["retrieve:miss"],
        )
    top = hits[0]
    return AssistantReply(
        answer=f"[{top.doc_id}] {top.text}",
        sources=[top.doc_id],
        used_memory=False,
        used_retrieval=True,
        trace=["retrieve:hit", f"cite:{top.doc_id}"],
    )


r = retrieval_only_assistant("alice", "t1", "What is the return window?")
print(r.answer[:80], "...")

---

## 6. Memory-Only Assistant

Uses stored user facts only. Can personalize but **cannot** cite official policy — dangerous for compliance questions.

In [ ]:
def memory_only_assistant(
    store: AgentMemoryStore, user_id: str, thread_id: str, message: str
) -> AssistantReply:
    apply_memory_writes(store, user_id, thread_id, message)
    mem = store.format_for_prompt(user_id, thread_id)
    order_id = store.get(user_id, "last_order_id")
    name = store.get(user_id, "customer_name")

    if "status" in message.lower() and order_id:
        row = ORDERS_DB.get(order_id)
        status = row["status"] if row else "unknown"
        answer = f"{name or 'Customer'}, order {order_id} status is {status} (from memory + order DB)."
    elif "return" in message.lower():
        answer = f"{name or 'Customer'}, I recall you asked about returns before, but I have no policy document loaded."
    else:
        answer = f"Hello {name or 'there'}. I remember: {mem}"

    return AssistantReply(
        answer=answer,
        sources=["memory"],
        used_memory=True,
        used_retrieval=False,
        trace=["memory:read", "memory:write"],
    )


m = memory_only_assistant(memory, "alice", "t1", "What is my order status?")
print(m.answer)

---

## 7. Hybrid Assistant — Memory + Retrieval

Production pattern: **memory** for who/what thread context; **retrieval** for authoritative policy text.

In [ ]:
def hybrid_assistant(
    store: AgentMemoryStore, user_id: str, thread_id: str, message: str
) -> AssistantReply:
    trace: list[str] = []
    apply_memory_writes(store, user_id, thread_id, message)
    trace.append("memory:write")

    mem_block = store.format_for_prompt(user_id, thread_id)
    trace.append("memory:read")

    # Build retrieval query — expand with memory context
    order_id = store.get(user_id, "last_order_id")
    retrieval_query = message
    if "it" in message.lower() and order_id and "order" not in message.lower():
        retrieval_query = f"{message} order {order_id}"
        trace.append(f"memory:augment_query→{retrieval_query}")

    hits = retrieve_policies(retrieval_query)
    trace.append(f"retrieve:{len(hits)} hits")

    name = store.get(user_id, "customer_name") or "there"
    parts: list[str] = [f"Hi {name},"]

    if hits and any(k in message.lower() for k in ("return", "refund", "policy", "shipping", "billing")):
        top = hits[0]
        parts.append(f"Policy [{top.doc_id}]: {top.text}")

    if order_id and ("status" in message.lower() or "it" in message.lower()):
        row = ORDERS_DB.get(order_id)
        if row:
            parts.append(f"Your order {order_id} is {row['status']}.")

    if len(parts) == 1:
        parts.append("How can I help with your ShopCo order or policy question?")

    sources = [h.doc_id for h in hits] + (["memory"] if mem_block != "(no user memory)" else [])
    return AssistantReply(
        answer=" ".join(parts),
        sources=sources,
        used_memory=True,
        used_retrieval=bool(hits),
        trace=trace,
    )


h = hybrid_assistant(memory, "alice", "t1", "Can I return it?")
print(h.answer)
print("Trace:", h.trace)

---

## 8. Multi-Turn Demo — Where Retrieval Alone Breaks

Turn 1 establishes order ID. Turn 2 says *"Can I return it?"* — pronoun needs **memory**, policy needs **retrieval**.

In [ ]:
def run_conversation(
    assistant_fn, user_id: str, turns: list[str], *, fresh_memory: bool = False
) -> list[AssistantReply]:
    store = AgentMemoryStore() if fresh_memory else memory
    thread_id = str(uuid.uuid4())[:8]
    replies: list[AssistantReply] = []
    for msg in turns:
        if assistant_fn == retrieval_only_assistant:
            replies.append(assistant_fn(user_id, thread_id, msg))
        else:
            replies.append(assistant_fn(store, user_id, thread_id, msg))
    return replies


TURNS = [
    "My name is Alice. I have order ORD-1001.",
    "Can I return it?",
]

print("=== Retrieval-only ===")
for i, r in enumerate(run_conversation(retrieval_only_assistant, "alice", TURNS, fresh_memory=True), 1):
    print(f"  Turn {i}: {r.answer[:85]}...")

print("\n=== Hybrid ===")
for i, r in enumerate(run_conversation(hybrid_assistant, "alice", TURNS, fresh_memory=True), 1):
    print(f"  Turn {i}: {r.answer[:85]}...")

---

## 9. Decision Matrix — What Goes Where

| Data | Store | Why |
|------|-------|-----|
| Return policy text | **Retrieval** | Shared, versioned, citable |
| User's name | **Memory (semantic)** | Per-user |
| Order ID mentioned turn 1 | **Memory (episodic)** | Thread context |
| Live order status | **Tool / SQL** | Not memory, not RAG |
| "User prefers email" | **Memory (semantic)** | Personalization |
| Full chat transcript | **Working memory / summary** | Context compression |
| Company org chart | **Retrieval** | Shared reference |

---

## 10. When Retrieval Alone Is Enough

Use **RAG only** (no long-term memory) when:

- Single-turn Q&A over static docs
- No personalization required
- Every answer must cite official sources
- Stateless API (no `user_id`)

Example: internal policy lookup bot with no conversation history.

In [ ]:
SINGLE_TURN_OK = [
    "What is the return window?",
    "When is shipping free?",
    "How do billing disputes work?",
]

for q in SINGLE_TURN_OK:
    r = retrieval_only_assistant("anon", "s1", q)
    ok = bool(r.sources) and r.sources[0].startswith("pol-")
    print(f"{'✓' if ok else '✗'} {q} → {r.sources}")

---

## 11. When Memory Is Required

Add **memory** when:

- Multi-turn dialogs with pronouns / ellipsis ("that order", "it")
- Personalization (name, locale, tier)
- Agent must remember user-stated constraints across sessions
- Retrieval query needs enrichment from prior turns

In [ ]:
MEMORY_REQUIRED = [
    ("Setup", "My order is ORD-1002."),
    ("Follow-up", "What's the status?"),
    ("Follow-up", "Can I still return it?"),
]

store = AgentMemoryStore()
thread = "mem-demo"
print(f"{'Turn':<12} {'Retrieval-only':<40} Hybrid")
print("-" * 90)
for label, msg in MEMORY_REQUIRED:
    ro = retrieval_only_assistant("bob", thread, msg)
    hy = hybrid_assistant(store, "bob", thread, msg)
    ro_has_order = "ORD-1002" in ro.answer
    hy_has_order = "ORD-1002" in hy.answer
    print(f"{label:<12} order_ref={ro_has_order!s:<35} order_ref={hy_has_order}")

---

## 12. Decision Tree

```
Is the fact in a shared document corpus?
  YES → Retrieval (RAG)
  NO  → continue

Is the fact specific to this user/session?
  YES → Memory
  NO  → Tool/API or abstain

Does the question need BOTH policy + user context?
  YES → Hybrid (memory enriches query + retrieval cites policy)

Is it live operational data (order status now)?
  YES → Tool/SQL — neither memory nor RAG is source of truth
```

---

## 13. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| Index user chats in RAG | Privacy leak; stale user facts in shared search | Per-user memory store |
| Store policies only in memory | No citations; can't update docs | Policy corpus in retrieval |
| Memory for order status | Becomes stale | Tool lookup each time |
| Retrieval-only multi-turn | Loses thread context | Episodic memory |
| Huge transcript in every prompt | Token bloat | Summarize to working memory |

---

## 14. Evaluation Harness

In [ ]:
SCENARIOS = [
    {
        "name": "single_turn_policy",
        "turns": ["What is the return window?"],
        "expect_policy_cite": True,
        "expect_order_ref": False,
    },
    {
        "name": "multi_turn_pronoun",
        "turns": ["Order ORD-1001 here.", "Can I return it?"],
        "expect_policy_cite": True,
        "expect_order_ref": True,
    },
]


def eval_assistant(fn, scenario: dict) -> bool:
    store = AgentMemoryStore()
    thread = "eval"
    last: AssistantReply | None = None
    for msg in scenario["turns"]:
        if fn == retrieval_only_assistant:
            last = fn("u", thread, msg)
        else:
            last = fn(store, "u", thread, msg)
    assert last is not None
    cite_ok = any(s.startswith("pol-") for s in last.sources) if scenario["expect_policy_cite"] else True
    order_ok = ("ORD-1001" in last.answer) if scenario["expect_order_ref"] else True
    return cite_ok and order_ok


print(f"{'Scenario':<25} {'Retrieval':<12} {'Hybrid':<12}")
print("-" * 50)
for sc in SCENARIOS:
    ro = eval_assistant(retrieval_only_assistant, sc)
    hy = eval_assistant(hybrid_assistant, sc)
    print(f"{sc['name']:<25} {str(ro):<12} {str(hy):<12}")

---

## 15. Architecture — Hybrid Agent Context Assembly

```
  user message
       │
       ├──────────────────► memory read (semantic + episodic)
       │                           │
       │                           ▼
       │                    augment retrieval query
       │                           │
       └──────────────────► policy retriever
                                   │
                                   ▼
              ┌────────────────────────────────────┐
              │  prompt = memory block             │
              │         + retrieved policy chunks  │
              │         + tool results (orders)  │
              └────────────────────────────────────┘
                                   │
                                   ▼
                              compose answer
                                   │
                                   ▼
                              memory write (extract facts)
```

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| "We have RAG, skip memory" | Fails on turn 2 pronouns | Add episodic memory |
| "Store everything in memory" | No policy citations | Retrieval for docs |
| Duplicate facts in both | Conflicts on update | Single source of truth per fact type |
| Never write memory | Always asks same questions | Extract after each turn |
| Memory without TTL | Stale preferences forever | Governance + decay |

---

## 17. Optional Live LLM Compose

Set `USE_LIVE_LLM = True` to synthesize hybrid answers with `gpt-4o-mini` from memory + retrieval blocks.

In [ ]:
def live_hybrid_compose(mem_block: str, hits: list[RetrievalHit], message: str) -> str:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    evidence = "\n".join(f"[{h.doc_id}] {h.text}" for h in hits)
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([
        SystemMessage(content="Use memory for user context and evidence for policy. Cite doc ids."),
        HumanMessage(content=f"Memory:\n{mem_block}\n\nEvidence:\n{evidence}\n\nUser: {message}"),
    ])
    return str(resp.content)


if USE_LIVE_LLM:
    s = AgentMemoryStore()
    s.write("alice", "t", "last_order_id", "ORD-1001", MemoryKind.EPISODIC)
    hits = retrieve_policies("return policy")
    print(live_hybrid_compose(s.format_for_prompt("alice", "t"), hits, "Can I return it?"))
else:
    print("Offline hybrid example:")
    s = AgentMemoryStore()
    print(hybrid_assistant(s, "carol", "t9", "My name is Carol. Order ORD-1002.").answer)

---

## 18. Quiz

1. Why should policy text live in retrieval, not long-term memory?
2. What goes wrong with retrieval-only on "Can I return it?" after turn 1?
3. Name two memory kinds and what each stores in ShopCo support.
4. When is hybrid (memory + retrieval) required vs retrieval alone?
5. Why is live order status neither memory nor RAG?

---

## 19. Summary

**Retrieval** grounds agents in **shared, citable documents**. **Memory** grounds agents in **user-specific, evolving context**. They solve different problems.

- **Retrieval alone** — single-turn policy lookup, stateless FAQ.
- **Memory alone** — personalization without authoritative docs (insufficient for compliance).
- **Hybrid** — multi-turn support: memory resolves *who/what* the user means; retrieval cites *what policy says*; tools fetch *live order state*.

Use the decision tree: shared doc → RAG; user/session fact → memory; live data → tool; often all three in production ShopCo-style agents.